# 02 · One-stage anchor-free detector: YOLOv8

Ultralytics YOLOv8 predicts boxes directly from grid locations (no anchors) and assigns positives with the
task-aligned assigner (TOOD). All experiments use the shared protocol below; each run is also turned into a
Kaggle submission.

In [ ]:
!pip install -q ultralytics
import os, urllib.request
# helper modules shared by all notebooks (data split, metric, submission format, torchvision training)
REPO = "https://raw.githubusercontent.com/MakcDutch/AI-homework2/main/"
for f in ["common.py", "tv_train.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(REPO + f, f)

In [ ]:
import time, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from ultralytics import YOLO
import common as C

ROOT = C.find_data_root()
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
os.makedirs(WORK, exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Protocol (same for every detector)
- train: every 5th frame of the 56 train clips (consecutive frames are near-duplicates, see 01_eda §6)
- validation: the 7 official val clips, **all** frames, COCO mAP from `common.coco_evaluate`
  (ignored regions as `iscrowd`, up to 500 detections per frame, like the leaderboard)
- ignored regions are painted grey in training frames
- fixed input size, epochs, batch and seed

In [ ]:
TRAIN_STEP = 5
IMGSZ = 1024
EPOCHS = 12
BATCH = 8
DEVICE = 0
DATA_DIR = "/tmp/yolo_data"  # outside /kaggle/working so the dataset is not saved as notebook output

EXPERIMENTS = {
    "yolov8s_base": dict(model="yolov8s.pt", imgsz=IMGSZ),
    "yolov8s_1280": dict(model="yolov8s.pt", imgsz=1280),                            # higher input resolution
    "yolov8s_p2":   dict(model="yolov8s-p2.yaml", weights="yolov8s.pt", imgsz=IMGSZ),  # extra stride-4 head for tiny objects
}
RUN = ["yolov8s_base"]  # one experiment per Kaggle session keeps each run well under the 12 h limit

In [ ]:
data_yaml = C.build_yolo_dataset(ROOT, DATA_DIR, train_step=TRAIN_STEP, val_step=5)
val_images = C.list_frames(ROOT, "val")["image"].tolist()
test_images = C.list_frames(ROOT, "test")["image"].tolist()
print(len(val_images), "val frames,", len(test_images), "test frames")

## Training, evaluation and submission

In [ ]:
def run_experiment(name, cfg):
    model = YOLO(cfg["model"])
    if "weights" in cfg:
        model = model.load(cfg["weights"])  # copies the pretrained layers that match the new architecture
    t0 = time.time()
    model.train(data=data_yaml, imgsz=cfg["imgsz"], epochs=EPOCHS, batch=BATCH, device=DEVICE, seed=C.SEED,
                workers=4, project=f"{WORK}/runs", name=name, exist_ok=True, verbose=False,
                **cfg.get("train_args", {}))
    train_hours = (time.time() - t0) / 3600
    run_dir = model.trainer.save_dir

    best = YOLO(f"{run_dir}/weights/best.pt")
    val_preds, ms = C.ultralytics_predict(best, val_images, imgsz=cfg["imgsz"], device=DEVICE)
    metrics, per_class = C.coco_evaluate(val_preds, ROOT, "val")
    test_preds, _ = C.ultralytics_predict(best, test_images, imgsz=cfg["imgsz"], device=DEVICE)

    C.save_preds(val_preds, f"{WORK}/val_preds_{name}.pkl")
    C.save_preds(test_preds, f"{WORK}/test_preds_{name}.pkl")
    C.make_submission(test_preds, ROOT, f"{WORK}/sub_{name}.csv")
    pd.Series(per_class, name=name).to_csv(f"{WORK}/per_class_{name}.csv")
    row = dict(exp=name, family="one-stage anchor-free", run_dir=str(run_dir), imgsz=cfg["imgsz"], epochs=EPOCHS,
               train_hours=round(train_hours, 2), ms_per_img=round(ms, 1), **metrics)
    C.results_table(f"{WORK}/results.csv", row)
    return row


for name in RUN:
    print(run_experiment(name, EXPERIMENTS[name]))

## Results

In [ ]:
display(C.results_table(f"{WORK}/results.csv").drop(columns="run_dir"))
per_class = pd.concat([pd.read_csv(f"{WORK}/per_class_{n}.csv", index_col=0) for n in RUN
                       if os.path.exists(f"{WORK}/per_class_{n}.csv")], axis=1)
display(per_class.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
results = C.results_table(f"{WORK}/results.csv").set_index("exp")
for name in RUN:
    log = pd.read_csv(f"{results.loc[name, 'run_dir']}/results.csv")
    log.columns = log.columns.str.strip()
    ax.plot(log["epoch"], log["metrics/mAP50-95(B)"], marker="o", label=name)
ax.set_xlabel("epoch"); ax.set_ylabel("val mAP50-95 (Ultralytics, val subset)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{WORK}/yolo_curves.png", dpi=120); plt.show()

## Where it fails
Ground truth (class colours), ignored regions (red hatch) and predictions with score ≥ 0.3 (white dashed)
on the val frames where the model misses most objects.

In [ ]:
name = RUN[-1]
preds = C.load_preds(f"{WORK}/val_preds_{name}.pkl")
C.show_failures(preds, ROOT, f"{WORK}/yolo_failures.png")